# S46_05 — LLM Evaluation

Evaluating LLMs is harder than evaluating classical ML — outputs are often free-form text without a single correct answer. Three evaluation paradigms:

1. **Reference-based metrics** — compare against gold-standard text (BLEU, ROUGE, METEOR)
2. **Benchmark suites** — standardized tests (MMLU, HellaSwag, HumanEval)
3. **LLM-as-judge** — use a strong LLM to evaluate outputs (scalable, flexible)

## Reference-based metrics

In [ ]:
# pip install evaluate rouge_score sacrebleu
import evaluate

# ROUGE — recall-oriented, used for summarization
rouge = evaluate.load('rouge')

predictions = [
    'The cat sat on the mat and looked around.',
    'Machine learning models learn from data.',
]
references = [
    'A cat was sitting on a mat.',
    'ML models are trained on data to make predictions.',
]

scores = rouge.compute(predictions=predictions, references=references)
for metric, value in scores.items():
    print(f'{metric}: {value:.3f}')
# ROUGE-1 = unigram overlap, ROUGE-2 = bigram overlap, ROUGE-L = longest common subsequence

In [ ]:
# BLEU — precision-oriented, used for translation
bleu = evaluate.load('bleu')

bleu_result = bleu.compute(
    predictions=['the cat sat on the mat'],
    references=[['the cat sat on the mat']],   # BLEU expects list of lists
)
print(f"BLEU: {bleu_result['bleu']:.3f}")

# BERTScore — embedding-based, captures semantic similarity better than n-gram overlap
# pip install bert-score
# bertscore = evaluate.load('bertscore')
# result = bertscore.compute(predictions=predictions, references=references, lang='en')
# print(f"BERTScore F1: {sum(result['f1'])/len(result['f1']):.3f}")

print('\nMetric guide:')
print('  ROUGE — summarization (recall matters)')
print('  BLEU  — translation (precision matters)')
print('  BERTScore — when paraphrases should score high')

## Benchmark suites

Reference-based metrics miss the point for instruction-following and reasoning tasks. Benchmarks provide curated multiple-choice or code tasks with known answers.

| Benchmark | Tests | What it measures |
|-----------|-------|------------------|
| MMLU | 57 subjects (law, math, medicine…) | General knowledge breadth |
| HellaSwag | Commonsense completion | Commonsense reasoning |
| HumanEval | 164 Python functions | Code generation |
| GSM8K | 8.5k grade-school math | Multi-step arithmetic |
| MATH | Competition math | Advanced reasoning |
| MT-Bench | Multi-turn chat scenarios | Conversational ability |

In [ ]:
# Run LLM-Eval Harness (Eleuther) locally:
# pip install lm-eval
# lm_eval --model hf --model_args pretrained=gpt2 --tasks hellaswag --limit 100

# Or evaluate with the datasets library directly
from datasets import load_dataset

# MMLU sample
mmlu = load_dataset('cais/mmlu', 'high_school_mathematics', split='test[:5]')
for example in mmlu:
    print(f"Q: {example['question']}")
    choices = ['A', 'B', 'C', 'D']
    for i, c in enumerate(example['choices']):
        print(f"  {choices[i]}: {c}")
    print(f"Answer: {choices[example['answer']]}")
    print()

## LLM-as-judge

Use a strong LLM (e.g. Claude Sonnet or GPT-4o) to evaluate model outputs. More flexible than metrics, more scalable than human evaluation.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

def llm_judge(question, answer, criteria=None):
    if criteria is None:
        criteria = ['accuracy', 'completeness', 'clarity']
    
    prompt = f"""You are evaluating an AI assistant's response to a question.

Question: {question}

Response to evaluate: {answer}

Rate the response on a scale of 1-5 for each criterion and explain briefly.
Respond ONLY with valid JSON:
{{{', '.join(f'"{c}": {{"score": int, "reason": str}}' for c in criteria)}}}"""
    
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return json.loads(msg.content[0].text)

question = 'What is gradient descent?'
answer = 'Gradient descent is an optimization algorithm that iteratively updates parameters by moving in the direction opposite to the gradient of the loss function.'

scores = llm_judge(question, answer)
print(json.dumps(scores, indent=2))

In [ ]:
# Pairwise evaluation — compare two model outputs
def pairwise_judge(question, response_a, response_b):
    prompt = f"""Compare two AI responses to the same question.

Question: {question}

Response A: {response_a}

Response B: {response_b}

Which response is better? Respond ONLY with JSON:
{{"winner": "A" | "B" | "tie", "reason": str, "confidence": "high" | "medium" | "low"}}"""
    
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return json.loads(msg.content[0].text)

result = pairwise_judge(
    question='Explain backpropagation.',
    response_a='Backpropagation computes gradients using the chain rule, flowing errors backward through the network.',
    response_b='It is when neural networks learn.',
)
print(json.dumps(result, indent=2))

## Evaluation pipeline

In [ ]:
import statistics

# Evaluate a set of question-answer pairs
eval_set = [
    {'q': 'What is overfitting?', 'a': 'When a model memorizes training data and fails to generalize to new data.'},
    {'q': 'What is cross-validation?', 'a': 'A technique that splits data into folds to estimate model performance.'},
    {'q': 'What is the bias-variance tradeoff?', 'a': 'The tradeoff between underfitting (high bias) and overfitting (high variance).'},
]

results = []
for item in eval_set:
    scores = llm_judge(item['q'], item['a'])
    avg_score = statistics.mean(v['score'] for v in scores.values())
    results.append({'question': item['q'][:40] + '...', 'avg_score': avg_score})

for r in results:
    print(f"{r['question']:45} score: {r['avg_score']:.1f}/5")

overall = statistics.mean(r['avg_score'] for r in results)
print(f'\nOverall average: {overall:.2f}/5')

## Evaluation framework summary

| Approach | Pros | Cons |
|----------|------|------|
| ROUGE/BLEU | Fast, deterministic | Misses paraphrases; poor for reasoning |
| BERTScore | Semantic similarity | Needs reference; can't judge correctness |
| Benchmarks | Comparable, reproducible | May not match your use case |
| LLM-as-judge | Flexible, scalable | Expensive; biased toward verbose answers |
| Human eval | Ground truth | Slow, expensive, inconsistent |

> **Best practice:** Use LLM-as-judge for development, a small human eval set for calibration, and track 1-2 benchmarks relevant to your use case for regression testing.

This completes S46. Next section: [S47_RAG](../S47_RAG/S47_01_why_rag.ipynb)